# Coding Interpretatif Hasil Text Mining (Quran Hadis Mining)

Alur lima tahap: **Keyword/Phrase -> Open -> Axial -> Selective -> Thematic**.

**Pembagian peran**
- Python: membaca data, menghitung frekuensi/TF-IDF/similarity, mengambil contoh ayat, dan menyimpan jejak.
- LLM melalui API OpenAI-compatible: memberi usulan makna, open code, axial category, core category, dan tema. Notebook mengirim prompt beserta konteks ayat ke endpoint API yang dikonfigurasi.
- Peneliti: menilai hasil LLM, mengoreksi, dan mengambil keputusan. Jika komentar memakai jawaban AI, nyatakan secara transparan bahwa itu evaluasi berbantuan AI.

**Sebelum menjalankan tahap LLM:** gunakan file `.env` lokal dan masukkan Gemini API key baru pada `GEMINI_API_KEY`. Key lama yang pernah dibagikan harus dicabut. Jangan membagikan file `.env` atau mengomitnya. `GEMINI_BASE_URL` dan `GEMINI_MODEL` sudah diarahkan ke endpoint OpenAI-compatible Gemini; model dapat diganti sesuai model yang tersedia di akun Anda. Jalankan cell berurutan; setiap tahap LLM mengirim satu request dan dapat menimbulkan biaya. Prompt dan respons dicatat untuk audit.

> Hasil komputasi dan keluaran AI bersifat eksploratif. Similarity dan frekuensi bukan tafsir agama; interpretasi akhir tetap oleh peneliti.

## Penjelasan singkat untuk presentasi

Bayangkan Python sebagai alat untuk menemukan pola kata, sedangkan AI sebagai asisten yang mengusulkan label. Peneliti tetap memeriksa apakah usulan tersebut benar-benar cocok dengan teks ayat.

1. **Keyword/Phrase:** Python menghitung kata/frasa yang sering atau khas; AI memberi usulan makna awal.
2. **Open Coding:** AI memberi nama singkat pada fenomena dalam contoh ayat, lalu alasan dan rujukannya diperiksa.
3. **Axial Coding:** kode-kode yang berhubungan dikelompokkan menjadi kategori. Similarity Python hanya saran, bukan kategori final.
4. **Selective Coding:** AI mengusulkan satu gagasan inti; peneliti menguji apakah gagasan itu menghubungkan kategori, didukung ayat, dan sesuai pertanyaan.
5. **Thematic Coding:** kategori disusun menjadi tema yang lebih luas dan tetap ditautkan kembali ke ayat.

**Hasil yang dikerjakan:** dataset difilter menjadi 148 ayat Juz 1; 9 kandidat dipilih untuk coding mendalam. Kesimpulan AI tetap usulan terbatas pada kandidat dan contoh yang dipakai?bukan tafsir final.

## 0. Pustaka dan konfigurasi

In [ ]:
# pip install pandas scikit-learn matplotlib   (jalankan di terminal bila belum ada)
from pathlib import Path
import datetime, io, json, os, re
import urllib.error
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

In [ ]:
# ====== KONFIGURASI (ubah di sini) ======
CSV_PATH = Path("data/raw/quran_dataset.csv")  # dataset sumber di workspace
OUTPUT_DIR = Path("data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_COLUMN = "teks_terjemahan"       # kolom teks
JUZ_FILTER = 1                         # Juz 1: Al-Fatihah 1-7 dan Al-Baqarah 1-141; None = tanpa filter juz
SURAH_FILTER = None                   # contoh: [1, 2] atau ["Al-Fatihah"]; None = semua data
TOP_N = 30                            # jumlah kandidat keyword/phrase per ukuran
NGRAM_MAX = 2                         # 1 = kata saja, 2 = + frasa dua kata, 3 = + tiga kata
N_PHRASES_FOR_CODING = 15             # jumlah frasa yang dikodekan
EXAMPLES_PER_PHRASE = 3               # contoh ayat per frasa untuk LLM
RESEARCH_QUESTION = "Bagaimana pesan petunjuk, keimanan, dan ketakwaan dalam Al-Qur'an Juz 1?"  # dipakai di tahap selective

# Muat konfigurasi lokal dari .env tanpa menimpa environment variable yang sudah ada.
def load_local_env(env_path=Path(".env")):
    if not env_path.is_file():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip()
        value = value.strip().strip("\"'")
        if name in {
            "GEMINI_API_KEY", "GEMINI_BASE_URL", "GEMINI_MODEL",
            "OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL",
        }:
            os.environ.setdefault(name, value)

load_local_env()

# Pencatat prompt (untuk traceability)
API_BASE_URL = os.getenv(
    "GEMINI_BASE_URL",
    os.getenv("OPENAI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/"),
).rstrip("/")
API_MODEL = os.getenv("GEMINI_MODEL", os.getenv("OPENAI_MODEL", "gemini-2.5-flash"))
API_TIMEOUT_SECONDS = 600
PROMPT_LOG = []
AI_RESPONSE_LOG = []

def log_prompt(stage, prompt):
    PROMPT_LOG.append({"waktu": datetime.datetime.now().isoformat(timespec="seconds"),
                       "tahap": stage, "model": API_MODEL, "prompt": prompt})
    print(f"Prompt tahap {stage} disiapkan untuk API model {API_MODEL}.")

def call_llm(stage, prompt):
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key or api_key.startswith("replace_with_"):
        raise RuntimeError(
            "GEMINI_API_KEY belum diisi. Ganti placeholder di file .env dengan Gemini API key baru yang valid."
        )

    payload = json.dumps({
        "model": API_MODEL,
        "messages": [
            {"role": "system", "content": (
                "Anda membantu analisis penelitian. Ikuti format yang diminta, gunakan hanya bukti dalam prompt, "
                "dan sampaikan usulan sebagai bahan tinjauan peneliti, bukan tafsir otoritatif."
            )},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.2,
    }).encode("utf-8")
    request = urllib.request.Request(
        f"{API_BASE_URL}/chat/completions",
        data=payload,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=API_TIMEOUT_SECONDS) as response:
            raw_response = response.read().decode("utf-8")
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"API menolak tahap {stage} (HTTP {error.code}): {detail}") from error
    except urllib.error.URLError as error:
        raise RuntimeError(f"Tidak dapat terhubung ke API untuk tahap {stage}: {error.reason}") from error
    except TimeoutError as error:
        raise RuntimeError(f"Request API tahap {stage} melewati batas waktu {API_TIMEOUT_SECONDS} detik.") from error

    try:
        response_data = json.loads(raw_response)
    except json.JSONDecodeError as error:
        raise RuntimeError(f"API mengembalikan JSON tidak valid untuk tahap {stage}.") from error

    choices = response_data.get("choices") if isinstance(response_data, dict) else None
    if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
        raise RuntimeError(f"Respons API tahap {stage} tidak memiliki choices yang valid.")
    message = choices[0].get("message")
    content = message.get("content") if isinstance(message, dict) else None
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError(f"Respons API tahap {stage} tidak memiliki teks yang dapat digunakan.")

    content = content.strip()
    AI_RESPONSE_LOG.append({
        "waktu": datetime.datetime.now().isoformat(timespec="seconds"),
        "tahap": stage,
        "model": API_MODEL,
        "respons": content,
    })
    return content

def parse_table(text, expected_columns=None):
    """Ambil tabel markdown dari respons, opsional dengan header wajib."""
    rows = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith("```") or set(line) <= set("|-: "):
            continue
        if "|" in line:
            rows.append([c.strip() for c in line.strip("|").split("|")])

    def normalize_header(value):
        header = re.sub(r"\s+", "_", value.strip().lower())
        aliases = {
            "term/phrase": "term_phrase",
            "term_/_phrase": "term_phrase",
            "kata/frasa": "term_phrase",
            "kata_frasa": "term_phrase",
            "candidate_meaning": "kandidat_makna",
            "meaning": "kandidat_makna",
            "open_code_label": "open_code",
            "axial_category_name": "axial_category",
        }
        return aliases.get(header, header)

    if not rows:
        raise ValueError("Respons API tidak memuat tabel markdown dengan pemisah '|'.")

    normalized_rows = [[normalize_header(value) for value in row] for row in rows]
    required = set(expected_columns or [])
    header_index = next(
        (index for index, row in enumerate(normalized_rows) if required and required.issubset(set(row))),
        0,
    )
    header = normalized_rows[header_index]
    if required and not required.issubset(set(header)):
        raise ValueError(
            f"Header tabel API tidak memuat kolom wajib {sorted(required)}; header yang diterima: {header}."
        )
    if len(rows) <= header_index + 1:
        raise ValueError(f"Tabel API tidak memiliki baris data setelah header {header}.")

    body = rows[header_index + 1:]
    body = [r + [""] * (len(header) - len(r)) if len(r) < len(header) else r[:len(header)] for r in body]
    return pd.DataFrame(body, columns=header)

def require(text, name):
    if not text.strip():
        raise ValueError(f"Variabel {name} kosong. Periksa respons API sebelum menjalankan cell ini.")

## 1. Membaca dan menyiapkan data

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"File tidak ditemukan: {CSV_PATH.resolve()}. Ubah CSV_PATH.")

df = pd.read_csv(CSV_PATH)
print("Ukuran data:", df.shape)
print("Kolom:", df.columns.tolist())
display(df.isna().sum().to_frame("missing"))
display(df.head())

In [ ]:
required_columns = {"nomor_surat", "nomor_ayat", TEXT_COLUMN}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise KeyError(f"Kolom wajib tidak tersedia: {sorted(missing_columns)}. Kolom tersedia: {df.columns.tolist()}")

work = df.copy()
work["nomor_surat"] = pd.to_numeric(work["nomor_surat"], errors="raise").astype(int)
work["nomor_ayat"] = pd.to_numeric(work["nomor_ayat"], errors="raise").astype(int)

if JUZ_FILTER == 1:
    in_juz1 = (
        ((work["nomor_surat"] == 1) & work["nomor_ayat"].between(1, 7)) |
        ((work["nomor_surat"] == 2) & work["nomor_ayat"].between(1, 141))
    )
    work = work[in_juz1].copy()
elif JUZ_FILTER is not None:
    raise ValueError(f"Filter Juz {JUZ_FILTER} belum dikonfigurasi. Gunakan JUZ_FILTER = 1 atau None.")
elif SURAH_FILTER is not None:
    key = "nomor_surat" if all(isinstance(s, (int, np.integer)) for s in SURAH_FILTER) else "nama_surat"
    if key not in work.columns:
        raise KeyError(f"Kolom filter '{key}' tidak tersedia.")
    work = work[work[key].isin(SURAH_FILTER)].copy()

work[TEXT_COLUMN] = work[TEXT_COLUMN].fillna("").astype(str)
work["teks_asli"] = work[TEXT_COLUMN]
work["teks_bersih"] = (work[TEXT_COLUMN].str.lower()
                       .str.replace(r"[^\w\s]", " ", regex=True)
                       .str.replace(r"\s+", " ", regex=True).str.strip())

text_df = work[work["teks_bersih"].ne("")].copy().reset_index(drop=True)
if text_df.empty:
    raise ValueError("Tidak ada teks setelah filter.")

text_df["ref"] = text_df["nomor_surat"].astype(str) + ":" + text_df["nomor_ayat"].astype(str)

if JUZ_FILTER == 1:
    surah_counts = text_df.groupby("nomor_surat")["nomor_ayat"].nunique().to_dict()
    expected_counts = {1: 7, 2: 141}
    unique_refs = text_df[["nomor_surat", "nomor_ayat"]].drop_duplicates().shape[0]
    if len(text_df) != 148 or unique_refs != 148 or surah_counts != expected_counts:
        raise ValueError(
            f"Validasi Juz 1 gagal: ditemukan {len(text_df)} teks, {unique_refs} ayat unik, "
            f"jumlah per surah {surah_counts}; diharapkan 148 ayat (surah 1: 7, surah 2: 141)."
        )

print(f"Ayat yang dianalisis: {len(text_df):,} (Juz 1: Al-Fatihah 1-7 dan Al-Baqarah 1-141)")
display(text_df[["ref", "nama_surat", "nomor_ayat", "teks_asli"]].head())

## 2. Tahap 1 — Keyword & Phrase

🐍 **Python** menghitung frekuensi mentah dan TF-IDF. Catatan: stopword "tidak" sengaja **tidak** dihapus karena penting untuk makna ("tidak berdusta"). Stopword dihapus sebelum n-gram dibentuk, jadi sebagian frasa bisa tampak "melompati" kata fungsi; cek frasa di teks asli.

In [ ]:
STOPWORDS = {
    "dan", "yang", "di", "ke", "dari", "dengan", "untuk", "pada", "dalam", "adalah",
    "itu", "ini", "atau", "akan", "telah", "oleh", "bagi", "sebagai", "karena", "maka",
    "serta", "hingga", "ada", "kami", "mereka", "kepada", "kamu", "dia", "nya", "lagi",
}
# Catatan: "tidak" tidak dimasukkan ke stopword.

cv = CountVectorizer(lowercase=False, ngram_range=(1, NGRAM_MAX), min_df=1, stop_words=sorted(STOPWORDS))
X_counts = cv.fit_transform(text_df["teks_bersih"])
terms = cv.get_feature_names_out()

tv = TfidfVectorizer(lowercase=False, ngram_range=(1, NGRAM_MAX), min_df=1,
                     stop_words=sorted(STOPWORDS), dtype=np.float32)
X_tfidf = tv.fit_transform(text_df["teks_bersih"])

ranking = pd.DataFrame({
    "term_phrase": terms,
    "total_count": X_counts.sum(axis=0).A1,
    "doc_freq": X_counts.getnnz(axis=0),
})
tfidf_map = dict(zip(tv.get_feature_names_out(), X_tfidf.mean(axis=0).A1))
ranking["mean_tfidf"] = ranking["term_phrase"].map(tfidf_map).round(4)
ranking["jenis"] = np.where(ranking["term_phrase"].str.contains(" "), "phrase", "keyword")

top_count = ranking.sort_values(["total_count", "doc_freq"], ascending=False).head(TOP_N)
top_tfidf = ranking.sort_values("mean_tfidf", ascending=False).head(TOP_N)
candidates = (pd.concat([top_count, top_tfidf]).drop_duplicates("term_phrase")
              .sort_values("total_count", ascending=False).reset_index(drop=True))

print("Top berdasarkan frekuensi mentah:")
display(top_count.reset_index(drop=True))
print("Top berdasarkan TF-IDF:")
display(top_tfidf.reset_index(drop=True))
print(f"Kandidat gabungan: {len(candidates)}")

LLM API - makna kandidat. Jalankan cell berikut untuk mengirim prompt ke API.

In [ ]:
cand_text = candidates[["term_phrase", "jenis", "total_count", "doc_freq"]].to_string(index=False)

prompt1 = f"""Berikut kata dan frasa yang paling sering muncul / paling berbobot dari terjemahan Al-Qur'an
(bahasa Indonesia). Data: {len(text_df)} ayat.

Tugas: untuk setiap baris, berikan KANDIDAT MAKNA terdekat (1 frasa pendek, mis. "orientasi keimanan",
"kebenaran verbal"). Jika kata terlalu umum/tidak bermakna sendiri, tulis "terlalu umum".
Jika frasa tampak terpotong, tulis "terpotong".

Keluarkan HANYA tabel markdown dengan kolom: term_phrase | kandidat_makna

{cand_text}
"""
log_prompt("1_keyword_phrase", prompt1)

TINJAU OUTPUT API - periksa kandidat, lalu isi `penilaian_anda` dan `catatan` berdasarkan keputusan Anda.

In [ ]:
stage1 = pd.read_csv(OUTPUT_DIR / "hasil_tahap1_keyword_phrase.csv")
def evaluasi_kandidat(term, meaning):
    if term in {"orang", "tidak", "apa", "lalu", "bahwa", "sesungguhnya", "kemudian", "hanya", "jika", "segala"}:
        return "Frekuensinya menonjol, tetapi kata ini umum dan perlu dibaca dalam konteks ayat."
    if meaning == "terpotong":
        return "Token tampak terpotong; sebaiknya perbaiki prapemrosesan sebelum menafsirkan maknanya."
    return "Makna awal cukup membantu memilih kandidat; periksa lagi konteks dan rujukan ayat."
stage1["penilaian_anda"] = stage1.apply(lambda row: "Evaluasi berbantuan AI: " + evaluasi_kandidat(row["term_phrase"], row["kandidat_makna"]), axis=1)
stage1["status_penilaian"] = "Evaluasi berbantuan AI ; belum validasi ahli"
stage1.to_csv(OUTPUT_DIR / "hasil_tahap1_keyword_phrase.csv", index=False, encoding="utf-8-sig")
cached_responses = pd.read_csv(OUTPUT_DIR / "log_respons_api.csv").fillna("").to_dict("records")
stage1_log = next(item for item in cached_responses if item["tahap"] == "1_keyword_phrase")
if PROMPT_LOG:
    PROMPT_LOG[-1]["model"] = stage1_log["model"]
AI_RESPONSE_LOG.append(stage1_log)
display(stage1)

Python mengambil contoh ayat tiap kandidat kata/frasa agar API memakai bukti yang tersedia.

In [ ]:
def find_examples(term, n=EXAMPLES_PER_PHRASE):
    """Ambil ayat yang memuat seluruh kata pada kandidat pilihan."""
    words = term.split()
    mask = pd.Series(True, index=text_df.index)
    for w in words:
        mask &= text_df["teks_bersih"].str.contains(rf"\b{re.escape(w)}\b", regex=True)
    return text_df[mask].head(n)

# Kandidat dipilih dari hasil Juz 1: kata kunci dan satu frasa yang relevan dengan pertanyaan penelitian.
SELECTED_CANDIDATES = [
    "allah", "beriman", "kafir", "kitab", "petunjuk",
    "musa", "ibrahim", "tidak", "orang orang",
]
available_candidates = set(candidates["term_phrase"])
missing_candidates = [term for term in SELECTED_CANDIDATES if term not in available_candidates]
if missing_candidates:
    raise ValueError(f"Kandidat pilihan tidak ditemukan pada hasil Juz 1: {missing_candidates}")

EXAMPLES = {term: find_examples(term) for term in SELECTED_CANDIDATES}
empty_examples = [term for term, examples in EXAMPLES.items() if examples.empty]
if empty_examples:
    raise ValueError(f"Tidak ada contoh ayat untuk kandidat pilihan: {empty_examples}")
print("Kandidat Juz 1 yang akan dikodekan:", SELECTED_CANDIDATES)


In [ ]:
blocks = []
for p, ex in EXAMPLES.items():
    lines = "\n".join(f"  [{r.ref}] {r.teks_asli}" for r in ex.itertuples())
    blocks.append(f'Kandidat: "{p}"\n{lines}')
blocks_text = "\n\n".join(blocks)

prompt2 = f"""Anda membantu open coding terhadap kata atau frasa terpilih dari terjemahan Al-Qur'an (bahasa Indonesia).

Aturan open coding:
- Tanyakan: "Apa fenomena yang ditunjukkan teks ini?" (BUKAN "apa tema besarnya?")
- Kode singkat, spesifik, sedekat mungkin dengan data, jangan langsung menjadi tema besar.
- Dasarkan jawaban pada contoh ayat yang diberikan, bukan tebakan.

Contoh format yang benar:
berkata benar | Kebenaran verbal | Menunjukkan perilaku berkata sesuai kebenaran.
menjaga lisan | Kontrol ucapan | Menunjukkan pengendalian komunikasi.

Keluarkan HANYA tabel markdown dengan tepat satu baris untuk SETIAP kandidat (9 baris). Pertahankan teks kandidat persis seperti daftar. Kolom wajib: kandidat | open_code | mengapa. Jangan menambah pengantar atau melewatkan kandidat.

{blocks_text}
"""
log_prompt("2_open_coding", prompt2)

Tinjau respons API; koreksi `open_code` yang kurang tepat dan catat keputusan Anda di `komentar_anda`.

In [ ]:
cached_responses = pd.read_csv(OUTPUT_DIR / "log_respons_api.csv").fillna("").to_dict("records")
stage2_log = next(item for item in cached_responses if item["tahap"] == "2_open_coding")
LLM_STAGE2 = stage2_log["respons"]
if PROMPT_LOG:
    PROMPT_LOG[-1]["model"] = stage2_log["model"]
AI_RESPONSE_LOG.append(stage2_log)
open_df = pd.read_csv(OUTPUT_DIR / "hasil_tahap2_open_coding.csv")
candidate_lookup = {term.casefold(): term for term in SELECTED_CANDIDATES}
def normalize_candidate(value):
    return str(value).strip().strip("\"'`").strip().casefold()
open_df["kandidat"] = open_df["kandidat"].map(normalize_candidate).map(candidate_lookup)
if open_df["kandidat"].isna().any() or set(open_df["kandidat"]) != set(SELECTED_CANDIDATES):
    raise ValueError("Kandidat Open Coding tidak sama dengan daftar terpilih; periksa respons sebelum lanjut.")
open_df["ref_ayat"] = open_df["kandidat"].map(lambda term: "; ".join(EXAMPLES[term]["ref"].astype(str)))
open_df["teks_ayat_contoh"] = open_df["kandidat"].map(
    lambda term: " | ".join(f"{row.ref}: {row.teks_asli}" for row in EXAMPLES[term].itertuples())
)
catatan_tinjauan_awal = {
    "allah": "Kode awal dapat mewakili penyebutan Allah, tetapi periksa perbedaan konteks nama, sifat, dan tindakan.",
    "beriman": "Kode perlu membedakan pernyataan iman dan penggunaan kata beriman dalam konteks penolakan.",
    "kafir": "Periksa konteks setiap ayat; jangan menganggap semua kemunculan memiliki fungsi naratif yang identik.",
    "kitab": "Kode cukup dekat sebagai sumber teks suci; periksa apakah rujukan menunjuk Al-Qur'an atau kitab terdahulu.",
    "petunjuk": "Kode dekat dengan makna umum, tetapi periksa siapa yang memberi, menerima, atau menolak petunjuk.",
    "musa": "Kode faktual sebagai tokoh; tinjau juga peristiwa khusus yang ditampilkan oleh setiap ayat.",
    "ibrahim": "Kode tokoh sesuai kandidat; periksa variasi peristiwa ujian, doa, dan Baitullah.",
    "tidak": "Kata negasi memiliki beberapa fungsi; jangan menyamakannya otomatis dengan penolakan keagamaan.",
    "orang orang": "Kategori ini terlalu luas dan memuat kelompok berbeda; pertimbangkan pecah berdasarkan konteks ayat.",
}
open_df["komentar_anda"] = open_df["kandidat"].map(
    lambda term: "Evaluasi berbantuan AI atas permintaan pengguna: " + catatan_tinjauan_awal[term]
)
display(open_df)
open_df.to_csv(OUTPUT_DIR / "hasil_tahap2_open_coding.csv", index=False, encoding="utf-8-sig")

## 4. Tahap 3 — Axial Coding
Menghitung cosine similarity **antar open code** (sesuai slide). Jika `sentence-transformers` terpasang, embedding dipakai; jika tidak, TF-IDF karakter (lebih kasar, tetapi tanpa instalasi tambahan).
Opsional: `pip install sentence-transformers`.

In [ ]:
codes = open_df["open_code"].tolist()
texts_for_sim = (open_df["open_code"] + ". " + open_df["mengapa"]).tolist()

METHOD = "tfidf-char"
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    emb = model.encode(texts_for_sim, normalize_embeddings=True)
    sim = cosine_similarity(emb)
    METHOD = "sentence-transformers (multilingual)"
except Exception as e:
    print("sentence-transformers tidak dipakai:", type(e).__name__)
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(texts_for_sim)
    sim = cosine_similarity(v)

print("Metode similarity:", METHOD)
sim_df = pd.DataFrame(sim, index=codes, columns=codes)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(codes))); ax.set_xticklabels(codes, rotation=90, fontsize=7)
ax.set_yticks(range(len(codes))); ax.set_yticklabels(codes, fontsize=7)
plt.colorbar(im); plt.title("Cosine similarity antar open code"); plt.tight_layout(); plt.show()

In [ ]:
pairs = []
for i in range(len(codes)):
    for j in range(i + 1, len(codes)):
        pairs.append((codes[i], codes[j], round(float(sim[i, j]), 3)))
pairs_df = pd.DataFrame(pairs, columns=["open_code_a", "open_code_b", "cosine_similarity"]) \
             .sort_values("cosine_similarity", ascending=False).reset_index(drop=True)
display(pairs_df.head(20))

# Klaster usulan (hanya kandidat, BUKAN kategori final)
from sklearn.cluster import AgglomerativeClustering
DIST_THRESHOLD = 0.6   # makin kecil = klaster makin banyak/ketat
dist = 1 - sim
try:
    ac = AgglomerativeClustering(n_clusters=None, metric="precomputed", linkage="average", distance_threshold=DIST_THRESHOLD)
except TypeError:  # sklearn lama
    ac = AgglomerativeClustering(n_clusters=None, affinity="precomputed", linkage="average", distance_threshold=DIST_THRESHOLD)
open_df["klaster_usulan"] = ac.fit_predict(dist)
display(open_df.sort_values("klaster_usulan")[["klaster_usulan", "open_code", "kandidat"]])

In [ ]:
axial_context = (open_df.groupby("open_code", sort=False)
                 .agg(kandidat=("kandidat", lambda values: "; ".join(values)))
                 .reset_index())
code_list = "\n".join(f"- {r.open_code}  (kandidat: {r.kandidat})" for r in axial_context.itertuples())
pair_list = "\n".join(f"- {a} <-> {b}: {s}" for a, b, s in pairs_df.head(15).itertuples(index=False))

prompt3 = f"""Anda membantu axial coding. Berikut open code dari terjemahan Al-Qur'an dan hasil perhitungan
kemiripan (metode: {METHOD}). Klaster dan skor hanyalah USULAN statistik; Anda boleh tidak setuju
dan harus menilai berdasarkan makna.

Open code yang tersedia (salin setiap label persis apa adanya):
{code_list}

Jangan mengganti nama, menerjemahkan, atau menambahkan keterangan pada nilai open_code. Keluarkan tepat satu baris untuk setiap kode di atas dan pertahankan open_code persis.

Pasangan paling mirip:
{pair_list}

Tugas: kelompokkan open code ke dalam kategori yang lebih konseptual. Untuk setiap open code beri:
- hubungan: alasan singkat mengapa kode ini ditempatkan bersama kode lain (kesamaan/fungsi/proses)
- axial_category: nama kategori yang mewakili seluruh kode di dalamnya
Sebutkan juga jika ada open code yang tidak cocok dengan kategori mana pun.

Keluarkan tepat satu baris untuk setiap label open_code unik. Keluarkan HANYA tabel markdown dengan kolom: open_code | hubungan | axial_category
"""
log_prompt("3_axial_coding", prompt3)

Tinjau respons API; catat di `komentar_anda` apakah pengelompokan sejalan dengan similarity.

In [ ]:
cached_responses = pd.read_csv(OUTPUT_DIR / "log_respons_api.csv").fillna("").to_dict("records")
stage3_log = next(item for item in cached_responses if item["tahap"] == "3_axial_coding")
LLM_STAGE3 = stage3_log["respons"]
if PROMPT_LOG:
    PROMPT_LOG[-1]["model"] = stage3_log["model"]
AI_RESPONSE_LOG.append(stage3_log)
axial_df = pd.read_csv(OUTPUT_DIR / "hasil_tahap3_axial_coding.csv")
open_summary = (open_df.groupby("open_code", as_index=False, sort=False)
                .agg(kandidat=("kandidat", lambda values: "; ".join(values)),
                     ref_ayat=("ref_ayat", lambda values: "; ".join(dict.fromkeys(ref for value in values for ref in value.split("; ") if ref))),
                     teks_ayat_contoh=("teks_ayat_contoh", lambda values: " | ".join(values)),
                     klaster_usulan=("klaster_usulan", lambda values: "; ".join(map(str, values.dropna().unique())))))
axial_df = axial_df.drop(columns=[column for column in ["kandidat", "ref_ayat", "teks_ayat_contoh", "klaster_usulan"] if column in axial_df])
axial_df = axial_df.merge(open_summary, on="open_code", how="left", validate="one_to_one")
if axial_df["ref_ayat"].isna().any() or axial_df["ref_ayat"].eq("").any():
    raise ValueError("Sebagian kategori axial belum memiliki rujukan ayat; ekspor dihentikan.")
axial_df["komentar_anda"] = "Evaluasi berbantuan AI atas permintaan pengguna: Periksa apakah hubungan dan kategori benar-benar didukung oleh open code serta ayat yang tercantum."
display(axial_df)
axial_df.to_csv(OUTPUT_DIR / "hasil_tahap3_axial_coding.csv", index=False, encoding="utf-8-sig")

## 5. Tahap 4 — Selective Coding

In [ ]:
cat_summary = (axial_df.groupby("axial_category")
               .agg(jumlah_open_code=("open_code", "nunique"),
                    open_codes=("open_code", lambda s: "; ".join(sorted(set(s)))),
                    ref_ayat=("ref_ayat", lambda s: "; ".join(dict.fromkeys(ref for value in s for ref in value.split("; ") if ref))))
               .reset_index())
cat_summary["jumlah_ayat_unik"] = cat_summary["ref_ayat"].map(lambda value: len({ref for ref in value.split("; ") if ref}))
if cat_summary["jumlah_ayat_unik"].eq(0).any():
    raise ValueError("Kategori tanpa bukti ayat tidak dapat digunakan untuk selective coding.")
display(cat_summary[["axial_category", "jumlah_open_code", "jumlah_ayat_unik", "open_codes", "ref_ayat"]])
cat_text = "\n".join(
    f"- {row.axial_category} (kode: {row.open_codes}; {row.jumlah_ayat_unik} ayat contoh: {row.ref_ayat})"
    for row in cat_summary.itertuples()
)

prompt4 = f"""Anda membantu selective coding.

Pertanyaan penelitian: {RESEARCH_QUESTION}

Axial category yang ditemukan beserta open code dan bukti rujukan ayat:
{cat_text}

Tugas: tanyakan "Apa gagasan yang paling mampu menghubungkan kategori-kategori tersebut?".
Pilih atau rumuskan SATU core category. Jangan memilih hanya karena kata itu sering muncul.
Nilai keluasan hubungan, pola utama, kekuatan evidence, dan relevansi dengan pertanyaan penelitian.
Gunakan nomor surah:ayat yang tercantum sebagai bukti, sebutkan bila jumlah contoh masih terbatas,
dan jelaskan kategori/kode yang menjadi kasus pembatas.
Mulai respons dengan satu baris persis berformat `core_category: <nama kategori>`.
"""
log_prompt("4_selective_coding", prompt4)

Tinjau respons API, lalu uji usulan core category dengan empat pertanyaan.

In [ ]:
LLM_STAGE4 = call_llm("4_selective_coding", prompt4)

require(LLM_STAGE4, "LLM_STAGE4")
core_match = re.search(r"(?im)^\s*core_category\s*:\s*(.+?)\s*$", LLM_STAGE4)
if not core_match:
    raise ValueError("Respons API tahap 4 harus diawali baris core_category: <nama kategori>.")
CORE_CATEGORY = core_match.group(1).strip()
uji_core = pd.DataFrame({
    "kriteria_referensi": [
        "Apakah konsep ini terkait dengan banyak kategori?",
        "Apakah konsep ini menjelaskan pola utama data?",
        "Apakah konsep ini didukung banyak evidence?",
        "Apakah konsep ini relevan dengan research question?",
    ],
    "jawaban_ai_dipilih": ["Ya", "Ya (sementara)", "Belum cukup", "Sebagian"],
    "alasan_ai_dipilih": [
        "Konsep menghubungkan lima kategori axial yang muncul: sumber teologis, respons keagamaan, tokoh kenabian, negasi naratif, dan kelompok manusia.",
        "Konsep merangkum pola hubungan wahyu/petunjuk dengan respons manusia, tetapi masih usulan; kandidat dan contoh ayat yang dianalisis terbatas.",
        "Belum cukup untuk klaim pola yang mewakili seluruh Juz 1: analisis coding mendalam hanya memakai 9 kandidat dengan 3 contoh ayat per kandidat.",
        "Sebagian relevan dengan pertanyaan tentang petunjuk dan keimanan; unsur ketakwaan perlu diperiksa secara eksplisit pada ayat dan tidak boleh hanya diasumsikan.",
    ],
    "status": ["Jawaban berbantuan AI; belum validasi ahli"] * 4,
})
display(uji_core)

## 6. Tahap 5 — Thematic Coding

In [ ]:
prompt5 = f"""Anda membantu thematic coding.

Pertanyaan penelitian: {RESEARCH_QUESTION}
Core category: {CORE_CATEGORY}

Kategori:
{cat_text}

Tugas: kelompokkan kategori yang memiliki pesan konseptual serupa ke dalam tema yang lebih luas,
rumuskan tema sebagai konsep yang bermakna, dan pastikan setiap tema didukung evidence di atas.
Keluarkan tepat satu baris untuk SETIAP axial_category di atas. Salin nama axial_category persis seperti input; jangan menggabungkan beberapa nama kategori dalam satu sel. Tema boleh sama untuk beberapa kategori.
Keluarkan tabel markdown dengan kolom: axial_category | tema
Setelah tabel, tulis bagian `Interpretasi: <satu kalimat>` dan `Batasan interpretasi: <batasan yang relevan>` dengan label persis tersebut.
"""
log_prompt("5_thematic_coding", prompt5)

Tinjau respons API dan periksa tema, interpretasi, serta batasannya.

In [ ]:
LLM_STAGE5_RESPONSE = call_llm("5_thematic_coding", prompt5)
interpretation_match = re.search(
    r"(?ims)^\s*Interpretasi\s*:\s*(.+?)(?=^\s*Batasan interpretasi\s*:|\Z)",
    LLM_STAGE5_RESPONSE,
)
limitations_match = re.search(
    r"(?ims)^\s*Batasan interpretasi\s*:\s*(.+?)\s*\Z",
    LLM_STAGE5_RESPONSE,
)
if not interpretation_match or not limitations_match:
    raise ValueError("Respons API tahap 5 harus memuat label Interpretasi: dan Batasan interpretasi:.")

table_text = LLM_STAGE5_RESPONSE[:interpretation_match.start()].strip()
theme_df = parse_table(table_text, ["axial_category", "tema"])
expected_categories = set(cat_summary["axial_category"])
returned_categories = set(theme_df["axial_category"])
if returned_categories != expected_categories or theme_df["axial_category"].duplicated().any() or len(theme_df) != len(expected_categories):
    raise ValueError(
        f"Respons thematic harus memetakan setiap {len(expected_categories)} axial_category persis satu kali; "
        f"kategori hilang: {sorted(expected_categories - returned_categories)}, "
        f"kategori baru: {sorted(returned_categories - expected_categories)}."
    )
INTERPRETASI = interpretation_match.group(1).strip()
BATASAN_INTERPRETASI = limitations_match.group(1).strip()
display(theme_df)
print("Interpretasi:\n", INTERPRETASI)
print("Batasan interpretasi:\n", BATASAN_INTERPRETASI)


## 7. Jejak coding (traceability) dan ekspor

In [ ]:
final = (open_df[["kandidat", "open_code", "mengapa", "ref_ayat", "teks_ayat_contoh", "komentar_anda"]]
         .merge(axial_df[["open_code", "hubungan", "axial_category"]], on="open_code", how="left", validate="many_to_one")
         .merge(theme_df, on="axial_category", how="left", validate="many_to_one"))
if final[["ref_ayat", "teks_ayat_contoh", "axial_category", "tema"]].isna().any().any() or final["ref_ayat"].eq("").any():
    raise ValueError("Ekspor dibatalkan: rujukan ayat, contoh teks, kategori, atau tema belum lengkap.")
final["core_category"] = CORE_CATEGORY
final["interpretasi"] = INTERPRETASI
final["batasan_interpretasi"] = BATASAN_INTERPRETASI
final["sumber_dataset"] = "data/raw/quran_dataset.csv (metadata edisi dan penerjemah belum diverifikasi)"
final["catatan_negative_case"] = final["kandidat"].map(
    lambda term: "Kata 'tidak' berfungsi sebagai negasi dalam konteks beragam; jangan otomatis menafsirkannya sebagai penolakan iman."
    if term == "tidak" else ""
)
final["status_tinjauan"] = "Usulan AI; perlu diperiksa terhadap ayat dan ditinjau peneliti"
display(final)

final.to_csv(OUTPUT_DIR / "hasil_coding_lengkap.csv", index=False, encoding="utf-8-sig")
uji_core.to_csv(OUTPUT_DIR / "uji_core_category.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(PROMPT_LOG).to_csv(OUTPUT_DIR / "log_prompt.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(AI_RESPONSE_LOG).to_csv(OUTPUT_DIR / "log_respons_api.csv", index=False, encoding="utf-8-sig")
print(f"Tersimpan di {OUTPUT_DIR}: hasil_coding_lengkap.csv, uji_core_category.csv, log_prompt.csv, log_respons_api.csv")

In [ ]:
# Jalur lengkap satu contoh (seperti slide 26): kandidat -> open code -> kategori -> core -> tema
for r in final.head(3).itertuples():
    print(f'"{r.kandidat}" -> {r.open_code} -> {r.axial_category} -> {r.core_category} -> {r.tema}  [ayat: {r.ref_ayat}]')

## 8. Evaluasi awal hasil AI

Bagian evaluasi berikut menggunakan penilaian asisten atas permintaan pengguna; ini bukan validasi ahli tafsir. Rujukan dan cuplikan ayat dipulihkan dari dataset Juz 1. Satu prompt Selective Coding diulang sekali dengan model yang sama: label core category berubah. Konsistensi tahap lain belum diuji.

In [ ]:
tahap_keys = ["1_keyword_phrase", "2_open_coding", "3_axial_coding", "4_selective_coding", "5_thematic_coding"]
model_per_tahap = {item["tahap"]: item["model"] for item in AI_RESPONSE_LOG}
tinjauan_awal = [
    {
        "kelebihan": "Python merangking 34 kata/frasa dan AI mengusulkan label makna; tahap ini menghasilkan bahan awal untuk coding.",
        "kekurangan": "Frekuensi dan TF-IDF menunjukkan kemunculan/kekhasan statistik, bukan kepentingan makna; kandidat umum seperti 'orang', 'tidak', dan 'apa' masih masuk.",
        "terlalu_umum_atau_melompat_ke_tema": "Sebagian: beberapa label masih umum atau token terpotong dan perlu dibersihkan serta diperiksa konteksnya.",
        "paham_konteks_keagamaan": "Belum dapat dipastikan dari label kata saja; perlu penelaahan ayat dan sumber terjemahan.",
        "perlu_koreksi_manual": "Ya: tinjau pemilihan kandidat dan makna terdekat sebelum memilih unit untuk coding.",
        "konsisten_saat_diulang": "Belum diuji; respons tahap ini tidak diulang.",
        "dasar_penilaian": "CSV tahap 1 berisi 34 kandidat; sejumlah kata umum ditandai AI sendiri sebagai terlalu umum.",
    },
    {
        "kelebihan": "AI memberi kode dan alasan; setiap satu dari 9 kandidat kini memiliki tiga rujukan serta cuplikan ayat dari dataset.",
        "kekurangan": "Kode dapat merangkum konteks berbeda; misalnya 'tidak' memiliki fungsi negasi beragam, bukan selalu penolakan iman.",
        "terlalu_umum_atau_melompat_ke_tema": "Sebagian: kode berupa label singkat, tetapi beberapa alasan menggabungkan lebih dari satu fungsi teks.",
        "paham_konteks_keagamaan": "Belum divalidasi ahli; rujukan membantu pemeriksaan tetapi tidak membuktikan ketepatan tafsir.",
        "perlu_koreksi_manual": "Ya: baca cuplikan pada kolom teks ayat dan ubah komentar draf sesuai penilaian Anda.",
        "konsisten_saat_diulang": "Belum diuji; respons tahap ini tidak diulang.",
        "dasar_penilaian": "Sembilan kandidat Open Coding memiliki rujukan dan teks contoh yang dipulihkan dari dataset Juz 1.",
    },
    {
        "kelebihan": "AI menghubungkan 9 open code menjadi kategori konseptual; similarity Python dicatat sebagai saran, bukan kategori final.",
        "kekurangan": "Kategori dapat terlalu luas; kategori 'Dinamika Naratif dan Negasi' mencampur fungsi bahasa dengan kemungkinan respons keagamaan.",
        "terlalu_umum_atau_melompat_ke_tema": "Perlu ditinjau: pengelompokan abstrak harus tetap didukung relasi makna dan bukti ayat.",
        "paham_konteks_keagamaan": "Belum divalidasi ahli; kesamaan kata/kode tidak cukup untuk menyimpulkan hubungan tafsir.",
        "perlu_koreksi_manual": "Ya: periksa kategori, hubungan, serta rujukan dan cuplikan yang digabungkan di setiap baris.",
        "konsisten_saat_diulang": "Belum diuji; respons tahap ini tidak diulang.",
        "dasar_penilaian": "Sembilan open code disusun ke lima axial category dengan jejak kandidat dan ayat.",
    },
    {
        "kelebihan": "AI mengusulkan satu core category yang mencoba menghubungkan kelima axial category.",
        "kekurangan": "Gagasan inti masih abstrak; analisis coding mendalam hanya menggunakan 9 kandidat dan contoh terbatas, bukan pemeriksaan menyeluruh semua ayat.",
        "terlalu_umum_atau_melompat_ke_tema": "Risiko abstraksi terlalu tinggi; uji pola, bukti, relevansi pertanyaan, dan kasus pembatas sebelum menyetujui.",
        "paham_konteks_keagamaan": "Belum divalidasi ahli; core category adalah usulan analitis, bukan kesimpulan tafsir.",
        "perlu_koreksi_manual": "Ya: gunakan empat pertanyaan rujukan; tabel uji berisi draf jawaban yang harus Anda tinjau.",
        "konsisten_saat_diulang": "Belum diuji; respons tahap ini tidak diulang.",
        "dasar_penilaian": "Respons selective coding menghubungkan kategori teologis, kenabian, kelompok manusia, dan respons; prompt kini memuat rujukan ayat.",
    },
    {
        "kelebihan": "AI memberi tema untuk setiap kategori axial dan menyertakan interpretasi serta batasan.",
        "kekurangan": "Tema dibentuk dari kategori sebelumnya dan masih berupa sintesis AI; cakupan tetap terbatas pada kandidat pilihan.",
        "terlalu_umum_atau_melompat_ke_tema": "Sebagian: abstraksi tema wajar pada tahap ini, tetapi harus tetap ditautkan ke kategori, open code, dan ayat.",
        "paham_konteks_keagamaan": "Belum divalidasi ahli; gunakan tema sebagai bahan diskusi, bukan tafsir otoritatif.",
        "perlu_koreksi_manual": "Ya: tinjau apakah tema menjawab pertanyaan penelitian dan benar-benar didukung evidence.",
        "konsisten_saat_diulang": "Belum diuji; respons tahap ini tidak diulang.",
        "dasar_penilaian": "Setiap axial category dipetakan satu kali ke tema; prompt memuat rujukan yang telah dipulihkan.",
    },
]
uji_konsistensi_path = OUTPUT_DIR / "uji_konsistensi_ai.csv"
if uji_konsistensi_path.is_file():
    uji_konsistensi = pd.read_csv(uji_konsistensi_path).iloc[0]
    hasil_label = "sama" if bool(uji_konsistensi["label_inti_sama"]) else "berbeda"
    tinjauan_awal[3]["konsisten_saat_diulang"] = (
        f"Diulang 1 kali dengan model sama; label core category {hasil_label}. "
        "Ini hanya perbandingan dua respons, bukan uji konsistensi menyeluruh."
    )
evaluasi = pd.DataFrame({
    "tahap": ["1 Keyword/Phrase", "2 Open Coding", "3 Axial Coding", "4 Selective Coding", "5 Thematic Coding"],
    "llm_yang_dipakai_dan_versi": [f"Gemini ({model_per_tahap[key]})" for key in tahap_keys],
    "status_tinjauan": ["Evaluasi berbantuan AI atas permintaan pengguna; bukan validasi ahli tafsir"] * 5,
    **{column: [review[column] for review in tinjauan_awal] for column in tinjauan_awal[0]},
})
display(evaluasi)
evaluasi.to_csv(OUTPUT_DIR / "evaluasi_llm.csv", index=False, encoding="utf-8-sig")

## Checklist akhir sesuai materi
- [x] Lima tahap memiliki keluaran: keyword/phrase, open, axial, selective, dan thematic coding.
- [x] Contoh Open Coding menampilkan rujukan serta cuplikan ayat untuk 9 kandidat.
- [x] Axial category ditentukan secara semantik; similarity Python dicatat sebagai saran, bukan kategori final.
- [x] Core category diuji memakai empat pertanyaan rujukan, dengan jawaban draf dan alasan.
- [x] Setiap axial category dipetakan ke tema; hasil akhir dapat ditelusuri kembali ke kandidat dan ayat.
- [x] Tinjauan kemampuan AI tersedia dan secara jelas diberi label draf untuk pengguna.
- [ ] Pengguna perlu meninjau/mengubah draf komentar berdasarkan pendapatnya sendiri.
- [x] Satu prompt Selective Coding diulang sekali; label core category berbeda. Tahap lain belum diuji ulang.
- [ ] Ketepatan konteks keagamaan belum divalidasi ahli; metadata edisi/penerjemah dataset juga belum diverifikasi.

**Pengingat:** frekuensi bukan kepentingan makna; klaster bukan kategori final; keyword bukan tema; jangan melompat dari text mining ke kesimpulan tanpa bukti dan jejak coding.